In [0]:
spark.sql("USE CATALOG ws_banking_etl2")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

transactions = spark.read.table("silver.transactions")


1. Which day(s) of the week sees the highest number of fraudulent transactions?

In [0]:
from pyspark.sql.functions import date_format, col, desc

fraud_by_weekday = (
    transactions
    .filter(col("is_fraud") == True)
    .groupBy(date_format("transaction_date", "EEEE").alias("day_of_week"))
    .count()
    .withColumnRenamed("count", "fraud_transaction_count")
    .orderBy(desc("fraud_transaction_count"))
)

fraud_by_weekday.show()
fraud_by_weekday.write.mode("overwrite").saveAsTable("gold.fraud_by_weekday")

+-----------+-----------------------+
|day_of_week|fraud_transaction_count|
+-----------+-----------------------+
|     Sunday|                   2646|
|     Friday|                   2284|
|   Thursday|                   2082|
|    Tuesday|                   2037|
|     Monday|                   1747|
|   Saturday|                   1434|
|  Wednesday|                   1102|
+-----------+-----------------------+



2. What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?

Fraud rate = fraud_transactions / total_transactions

In [0]:
from pyspark.sql.functions import date_format, avg, when

fraud_rate_monthly = (
    transactions
    .groupBy(date_format("transaction_date", "yyyy-MM").alias("month"))
    .agg(
        avg(when(col("is_fraud"), 1).otherwise(0)).alias("fraud_rate")
    )
    .orderBy("month")
)

fraud_rate_monthly.show(5)
fraud_rate_monthly.write.mode("overwrite").saveAsTable("gold.fraud_rate_monthly")

+-------+--------------------+
|  month|          fraud_rate|
+-------+--------------------+
|2010-01|0.001057218231580...|
|2010-02|0.002770942548411255|
|2010-03|0.002525521312109923|
|2010-04|0.002366001457536...|
|2010-05|0.002615177574375...|
+-------+--------------------+
only showing top 5 rows


3. Which users have the largest number of flagged (“is_fraud = true”) transactions?

In [0]:
top_fraud_users = (
    transactions
    .filter(col("is_fraud"))
    .groupBy("client_id")
    .count()
    .withColumnRenamed("count", "fraud_transaction_count")
    .orderBy(desc("fraud_transaction_count"))
)

top_fraud_users.show(5)
top_fraud_users.write.mode("overwrite").saveAsTable("gold.top_fraud_users")

+---------+-----------------------+
|client_id|fraud_transaction_count|
+---------+-----------------------+
|     1102|                     58|
|      209|                     52|
|       27|                     45|
|      155|                     44|
|     1128|                     43|
+---------+-----------------------+
only showing top 5 rows


4. Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:
from pyspark.sql.functions import weekofyear, year, avg, max, col, round

weekly_avg = (
    transactions
    .groupBy(
        "client_id",
        year(col("transaction_date")).alias("yr"),
        weekofyear(col("transaction_date")).alias("week")
    )
    .agg(
        avg("amount").alias("weekly_avg_amount"),
        max("amount").alias("weekly_max_amount")
    )
)

spike_users = (
    weekly_avg
    .filter(col("weekly_max_amount") > col("weekly_avg_amount") * 2)
    # shows how many times the max exceeds the average
    .withColumn("spike_ratio", round(col("weekly_max_amount") / col("weekly_avg_amount"), 2))
    .orderBy(col("spike_ratio").desc())
)

spike_users.show(5)
spike_users.write.mode("overwrite").saveAsTable("gold.users_amount_spikes")

+---------+----+----+-----------------+-----------------+-----------+
|client_id|  yr|week|weekly_avg_amount|weekly_max_amount|spike_ratio|
+---------+----+----+-----------------+-----------------+-----------+
|     1766|2018|  44|       0.00454545|          93.7100|   20616.22|
|     1737|2019|  35|       0.01714286|         172.6000|   10068.33|
|     1267|2011|  42|       0.02000000|         184.6500|    9232.50|
|      734|2012|  51|       0.01071429|          83.0000|    7746.66|
|     1666|2013|   9|       0.02666667|         193.9000|    7271.25|
+---------+----+----+-----------------+-----------------+-----------+
only showing top 5 rows


5. Which merchant categories exhibit the highest fraud rate?

In [0]:
from pyspark.sql.functions import count, sum, when, desc
fraud_rate_by_mcc = (
    transactions
        .groupBy("mcc", "mcc_description")
        .agg(
            count("*").alias("total_transactions"),
            sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
        )
        .withColumn(
            "fraud_rate",
            col("fraud_transactions") / col("total_transactions")
        )
        .orderBy(desc("fraud_rate"))
)

fraud_rate_by_mcc.show(5)
fraud_rate_by_mcc.write.mode("overwrite").saveAsTable("gold.fraud_rate_by_mcc")

+----+--------------------+------------------+------------------+-------------------+
| mcc|     mcc_description|total_transactions|fraud_transactions|         fraud_rate|
+----+--------------------+------------------+------------------+-------------------+
|4411|        Cruise Lines|               428|               165| 0.3855140186915888|
|5733|Music Stores - Mu...|               319|                76|0.23824451410658307|
|3006|Miscellaneous Fab...|               351|                29|0.08262108262108261|
|5045|Computers, Comput...|              2793|               204|0.07303974221267455|
|3144|Floor Covering St...|               334|                23| 0.0688622754491018|
+----+--------------------+------------------+------------------+-------------------+
only showing top 5 rows


6. Are there specific merchants with unusually high fraud volume?

In [0]:
high_fraud_merchants = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("merchant_id")
        .count()
        .withColumnRenamed("count", "fraud_transaction_count")
        .orderBy(desc("fraud_transaction_count"))
)

high_fraud_merchants.show(5)
high_fraud_merchants.write.mode("overwrite").saveAsTable("gold.high_fraud_merchants")

+-----------+-----------------------+
|merchant_id|fraud_transaction_count|
+-----------+-----------------------+
|      60569|                    764|
|      27092|                    725|
|      48919|                    332|
|      99370|                    302|
|      32858|                    289|
+-----------+-----------------------+
only showing top 5 rows


7. What’s the average transaction amount for fraud vs non-fraud transactions?

In [0]:
avg_amount = (
    transactions
    .groupBy("is_fraud")
    .agg(avg("amount").alias("avg_transaction_amount"))
)

avg_amount.show()
avg_amount.write.mode("overwrite").saveAsTable("gold.avg_amount")

+--------+----------------------+
|is_fraud|avg_transaction_amount|
+--------+----------------------+
|   false|           42.90858094|
|    true|          110.23468197|
+--------+----------------------+



8. Which merchant category has the highest total fraud amount?

In [0]:
mcc_fraud_amount = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("mcc", "mcc_description")
        .agg(sum("amount").alias("total_fraud_amount"))
        .orderBy(desc("total_fraud_amount"))
)

mcc_fraud_amount.show(5)
mcc_fraud_amount.write.mode("overwrite").saveAsTable("gold.mcc_highest_fraud_amount")

+----+-----------------+------------------+
| mcc|  mcc_description|total_fraud_amount|
+----+-----------------+------------------+
|5311|Department Stores|       225647.1900|
|4411|     Cruise Lines|       185946.7800|
|5300|  Wholesale Clubs|       113827.6500|
|5310|  Discount Stores|        81214.8900|
|4829|   Money Transfer|        66101.5200|
+----+-----------------+------------------+
only showing top 5 rows


9. What are the total monetary losses due to fraud each day?

In [0]:
from pyspark.sql.functions import to_date

daily_fraud_losses = (
    transactions
        .filter(col("is_fraud") == True)
        .withColumn("transaction_date", to_date(col("transaction_date")))
        .groupBy("transaction_date")
        .agg(sum("amount").alias("total_fraud_loss"))
        .orderBy(desc("total_fraud_loss"))
)

daily_fraud_losses.show(5)
daily_fraud_losses.write.mode("overwrite").saveAsTable("gold.daily_fraud_losses")

+----------------+----------------+
|transaction_date|total_fraud_loss|
+----------------+----------------+
|      2010-11-14|       7266.5900|
|      2010-02-19|       6684.2500|
|      2013-09-28|       6611.1200|
|      2010-03-26|       6491.0500|
|      2013-10-19|       6427.6300|
+----------------+----------------+
only showing top 5 rows


10. How many unique users commit fraudulent transactions per week?

In [0]:
from pyspark.sql.functions import weekofyear, year, countDistinct

weekly_unique_fraud_users = (
    transactions
        .filter(col("is_fraud") == True)
        .withColumn("year", year("transaction_date"))
        .withColumn("week", weekofyear("transaction_date"))
        .groupBy("year", "week")
        .agg(
            countDistinct("client_id").alias("unique_fraud_users"),
            count("*").alias("fraud_transaction_count")
        )
        .orderBy("year", "week")
)

weekly_unique_fraud_users.show(5)
weekly_unique_fraud_users.write.mode("overwrite").saveAsTable("gold.weekly_unique_fraud_users")

+----+----+------------------+-----------------------+
|year|week|unique_fraud_users|fraud_transaction_count|
+----+----+------------------+-----------------------+
|2010|   1|                 5|                     15|
|2010|   2|                 5|                     11|
|2010|   3|                 7|                     18|
|2010|   4|                17|                     61|
|2010|   5|                16|                     70|
+----+----+------------------+-----------------------+
only showing top 5 rows


11. Do fraud patterns show seasonal or monthly spikes?

In [0]:
from pyspark.sql.functions import month

monthly_fraud_trends = (
    transactions
        .withColumn("year", year("transaction_date"))
        .withColumn("month", month("transaction_date"))
        .groupBy("year", "month")
        .agg(
            count("*").alias("total_transactions"),
            sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions"),
            sum(when(col("is_fraud") == True, col("amount")).otherwise(0)).alias("total_fraud_amount")
        )
        .withColumn(
            "fraud_rate",
            col("fraud_transactions") / col("total_transactions")
        )
        .orderBy("year", "month")
)

monthly_fraud_trends.show(5)
monthly_fraud_trends.write.mode("overwrite").saveAsTable("gold.monthly_fraud_trends")

+----+-----+------------------+------------------+------------------+--------------------+
|year|month|total_transactions|fraud_transactions|total_fraud_amount|          fraud_rate|
+----+-----+------------------+------------------+------------------+--------------------+
|2010|    1|            101209|               107|        12253.5700|0.001057218231580...|
|2010|    2|             93470|               259|        32488.1100|0.002770942548411255|
|2010|    3|            103345|               261|        37454.2000|0.002525521312109923|
|2010|    4|            100169|               237|        29130.4200|0.002366001457536...|
|2010|    5|            104773|               274|        27258.8600|0.002615177574375...|
+----+-----+------------------+------------------+------------------+--------------------+
only showing top 5 rows


12. How has user behavior changed before versus after a fraudulent event?

In [0]:
from pyspark.sql.functions import min, when, avg, col

first_fraud = (
    transactions
        .filter(col("is_fraud") == True)
        .groupBy("client_id")
        .agg(min("transaction_date").alias("first_fraud_date"))
)

user_behavior_change = (
    transactions
        .join(first_fraud, "client_id")
        .withColumn(
            "period",
            when(col("transaction_date") < col("first_fraud_date"), "before_fraud")
            .otherwise("after_fraud")
        )
        .groupBy("client_id", "period")
        .agg(avg("amount").alias("avg_amount"))
)

user_behavior_change.show(5)
user_behavior_change.write.mode("overwrite").saveAsTable("gold.user_behavior_change")

+---------+------------+-----------+
|client_id|      period| avg_amount|
+---------+------------+-----------+
|     1967| after_fraud|36.44865505|
|      237|before_fraud|61.42297430|
|     1411|before_fraud|52.33035773|
|      984|before_fraud|58.19873518|
|     1082|before_fraud|56.20544383|
+---------+------------+-----------+
only showing top 5 rows


13. Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
value_segmented = (
    transactions
        .withColumn(
            "value_segment",
            when(col("amount") < 50, "Low")
            .when((col("amount") >= 50) & (col("amount") <= 200), "Medium")
            .otherwise("High")
        )
)

fraud_by_value = (
    value_segmented
        .groupBy("value_segment")
        .agg(
            count("*").alias("total_transactions"),
            sum(when(col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
        )
        .withColumn(
            "fraud_rate",
            col("fraud_transactions") / col("total_transactions")
        )
        .orderBy(desc("fraud_rate"))
)

fraud_by_value.show()
fraud_by_value.write.mode("overwrite").saveAsTable("gold.fraud_by_value_segment")

+-------------+------------------+------------------+--------------------+
|value_segment|total_transactions|fraud_transactions|          fraud_rate|
+-------------+------------------+------------------+--------------------+
|         High|            319101|              2194|0.006875566043353045|
|       Medium|           4139987|              5707|0.001378506744103...|
|          Low|           8846827|              5431|6.138924158910308E-4|
+-------------+------------------+------------------+--------------------+

